In [10]:
import torch
import torch.nn as nn
from torch_geometric.utils import scatter

class NeuroSAT(nn.Module):
    def __init__(self, d=128, num_iters=32):
        """
        Args:
            d (int): Embedding dimension for literals and clauses.
            num_iters (int): Number of message passing iterations.
        """
        super(NeuroSAT, self).__init__()
        self.d = d
        self.num_iters = num_iters

        # 1. Learnable initial embeddings for Literals and Clauses
        self.L_init = nn.Parameter(torch.randn(1, d))
        self.C_init = nn.Parameter(torch.randn(1, d))

        # 2. Message Passing MLPs
        self.mlp_l = nn.Sequential(
            nn.Linear(d, d),
            nn.ReLU(),
            nn.Linear(d, d)
        )
        self.mlp_c = nn.Sequential(
            nn.Linear(d, d),
            nn.ReLU(),
            nn.Linear(d, d)
        )

        # Optional but highly recommended: LayerNorm for training stability
        self.ln_l = nn.LayerNorm(d)
        self.ln_c = nn.LayerNorm(d)

        # 3. Update LSTMs
        self.lstm_c = nn.LSTMCell(d, d)
        # Literal LSTM takes [aggregated_messages, flipped_literal_embedding]
        self.lstm_l = nn.LSTMCell(2 * d, d) 

        # 4. Readout / Voting Network
        self.vote_mlp = nn.Sequential(
            nn.Linear(d, d),
            nn.ReLU(),
            nn.Linear(d, d),
            nn.ReLU(),
            nn.Linear(d, 1)
        )

    def forward(self, num_vars, num_clauses, edge_index, batch_lit=None):
        """
        Args:
            num_vars (int): Total number of variables.
            num_clauses (int): Total number of clauses.
            edge_index (Tensor): Shape [2, E]. Row 0 is literal indices, Row 1 is clause indices.
            batch_lit (Tensor, optional): Defines which formula in the batch each literal belongs to.
        Returns:
            logits (Tensor): SAT prediction logit per formula.
            votes (Tensor): Raw scalar vote per literal.
        """
        num_lits = num_vars * 2

        # Initialize hidden and cell states for all nodes
        L_h = self.L_init.repeat(num_lits, 1)
        L_c = torch.zeros_like(L_h)
        
        C_h = self.C_init.repeat(num_clauses, 1)
        C_c = torch.zeros_like(C_h)

        lit_idx, cls_idx = edge_index[0], edge_index[1]

        # Iterative Message Passing
        for _ in range(self.num_iters):
            
            # --- Phase 1: Literals -> Clauses ---
            L_msg = self.mlp_l(L_h)
            L_msg = self.ln_l(L_msg)
            
            # FIX: Gather the literal messages onto the edges first
            edge_msg_L = L_msg[lit_idx] 
            
            # Aggregate messages at target clauses
            C_agg = scatter(edge_msg_L, cls_idx, dim=0, dim_size=num_clauses, reduce='sum')
            
            # Update clause embeddings
            C_h, C_c = self.lstm_c(C_agg, (C_h, C_c))

            # --- Phase 2: Clauses -> Literals ---
            C_msg = self.mlp_c(C_h)
            C_msg = self.ln_c(C_msg)
            
            # FIX: Gather the clause messages onto the edges first
            edge_msg_C = C_msg[cls_idx] 
            
            # Aggregate messages at target literals
            L_agg = scatter(edge_msg_C, lit_idx, dim=0, dim_size=num_lits, reduce='sum')
            
            # Efficiently fetch the flipped literal embeddings (\bar{L})
            L_flip = L_h.view(-1, 2, self.d).flip(1).view(-1, self.d)
            
            # Concatenate aggregated messages with the flipped literal embedding
            L_input = torch.cat([L_agg, L_flip], dim=-1)
            
            # Update literal embeddings
            L_h, L_c = self.lstm_l(L_input, (L_h, L_c))

        # --- Phase 3: Readout ---
        # Generate a scalar vote for each literal
        votes = self.vote_mlp(L_h).squeeze(-1) # Shape: [num_lits]
        
        # Aggregate literal votes to compute formula satisfiability prediction
        if batch_lit is not None:
            logits = scatter(votes, batch_lit, dim=0, reduce='mean')
        else:
            logits = votes.mean().unsqueeze(0)
            
        return logits, votes

In [15]:
import torch
import torch.nn as nn
# Assuming the NeuroSAT class from the previous code block is already defined above this

def parse_dimacs_cnf(filepath):
    """
    Parses a DIMACS CNF file robustly, handling hidden characters and stray tokens.
    Returns the PyG edge_index and graph dimensions.
    """
    num_vars = 0
    num_clauses = 0
    
    edges_lit = []
    edges_cls = []
    
    current_clause_idx = 0
    
    # 'utf-8-sig' automatically safely strips hidden Byte Order Marks (BOM)
    with open(filepath, 'r', encoding='utf-8-sig') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            
            # Skip comments, empty lines, and standard DIMACS EOF markers (like '%')
            if not line or line.startswith('c') or line.startswith('%'):
                continue
                
            # Parse the problem definition line: "p cnf <vars> <clauses>"
            if line.startswith('p'):
                parts = line.split()
                num_vars = int(parts[2])
                num_clauses = int(parts[3])
                continue
                
            # Parse clauses
            tokens = line.split()
            for token in tokens:
                try:
                    # Attempt to parse the token into an integer
                    lit = int(token)
                except ValueError:
                    # If it fails (e.g., stray letter or EOF marker), print a warning and skip it
                    print(f"Warning: Skipping invalid integer '{token}' on line {line_num}")
                    continue
                
                if lit == 0:
                    # '0' marks the end of the current clause
                    current_clause_idx += 1
                else:
                    # Convert 1-indexed DIMACS variable to 0-indexed PyTorch variable
                    var_idx = abs(lit) - 1
                    is_positive = (lit > 0)
                    
                    # Map to literal nodes: 2*i for positive, 2*i+1 for negative
                    node_idx = 2 * var_idx if is_positive else 2 * var_idx + 1
                    
                    edges_lit.append(node_idx)
                    edges_cls.append(current_clause_idx)
                    
    # Build the edge index [2, E]
    # Row 0: Literal Nodes, Row 1: Clause Nodes
    edge_index = torch.tensor([edges_lit, edges_cls], dtype=torch.long)
    
    # Sanity check (changed to a warning rather than an assert, as some files are messy)
    if current_clause_idx != num_clauses:
         print(f"Warning: Header expected {num_clauses} clauses, but found {current_clause_idx}.")
    
    return num_vars, num_clauses, edge_index

# ==========================================
# Execution Example
# ==========================================

filepath = '/home/krishnendu/Research/fv-invariant-mining/data/circuits/cnf/miter_addr_mult_12bit.cnf'

# 2. Parse the file into graph tensors
num_vars, num_clauses, edge_index = parse_dimacs_cnf(filepath)

print(f"Parsed {filepath}:")
print(f"Variables: {num_vars}, Clauses: {num_clauses}")
print(f"Edge Index Shape: {edge_index.shape}")
print(f"Edge Index:\n{edge_index}\n")

# 3. Pass it to the NeuroSAT Model
model = NeuroSAT(d=1024, num_iters=10)

# Move everything to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
edge_index = edge_index.to(device)

# Forward pass
model.eval()
with torch.no_grad():
	logits, literal_votes = model(num_vars, num_clauses, edge_index)

# Convert logit to a SAT probability
sat_probability = torch.sigmoid(logits).item()
print(f"Predicted SAT Probability: {sat_probability:.4f}")

Parsed /home/krishnendu/Research/fv-invariant-mining/data/circuits/cnf/miter_addr_mult_12bit.cnf:
Variables: 60, Clauses: 168
Edge Index Shape: torch.Size([2, 476])
Edge Index:
tensor([[  2,  70,  95,   2,  71,  94,   3,  71,  95,   3,  70,  94,   4,   6,
           9,   4,   7,   8,   5,   7,   9,   5,   6,   8,   6,  71,  95,   7,
          70,   7,  94,   8,  72,  97,   8,  73,  96,   9,  73,  97,   9,  72,
          96,  10,  13,  14,  10,  12,  15,  11,  12,  14,  11,  13,  15,  12,
          74,  99,  12,  75,  98,  13,  75,  99,  13,  74,  98,  15,  72,   6,
          15,  96,   6,  15,  72,  96,  14,  73,  97,  14,  97,   7,  14,  73,
           7,  16,  19,  20,  16,  18,  21,  17,  18,  20,  17,  19,  21,  18,
          76, 101,  18,  77, 100,  19,  77, 101,  19,  76, 100,  21,  98,  14,
          21,  74,  98,  21,  74,  14,  20,  75,  99,  20,  99,  15,  20,  75,
          15,  22,  25,  26,  22,  24,  27,  23,  24,  26,  23,  25,  27,  24,
          78, 103,  24,  79, 102,